### Methods

#### **K-Sil Clustering Algorithm**
We use K-Sil (https://arxiv.org/abs/2506.12878) for narrative clustering.

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import euclidean_distances
from joblib import Parallel, delayed

class KSil:
    def __init__(self,
                 n_clusters=3,
                 init='random',
                 max_iter=100,
                 random_state=0,
                 init_temperature=1.0,
                 learning_rate=0.2,
                 tol=1e-4,
                 n_init=1, n_jobs=1):

        # Parameters
        self.n_clusters = int(n_clusters)     # Number of clusters
        self.init = init                      # Initialization method
        self.max_iter = int(max_iter)         # Maximum number of iterations
        self.init_temperature = float(init_temperature) # Initial temperature
        self.random_state = int(random_state) # Random seed
        self.learning_rate = learning_rate    # Learning rate for temperature
        self.tol = float(tol)                 # Centroid convergence tolerance

        # For multiple initializations:
        self.n_init = n_init                  # Number of initializations
        self.n_jobs = n_jobs                  # Number of cores

        # n_clusters check
        if self.n_clusters < 2:
           raise ValueError(f"n_clusters ({self.n_clusters}) must be > 1.")
        # init check
        if isinstance(self.init, str) and self.init not in ('random', 'k-means++'):
           raise ValueError(f"init ({self.init}) must be 'random' or 'k-means++'.")
        # n_init check
        if self.n_init < 1:
           raise ValueError(f"n_init ({self.n_init}) must be > 0.")

        # Attributes
        self.labels_ = None          # Cluster labels (converged partition)
        self.cluster_centers_ = None # Cluster centers (converged partition)
        self.n_iter_ = None          # Number of iterations (until convergence)
        self.sil_ = None             # silhouette (centroid-approximated) at converged partition

        # history attributes
        self.centers_history_ = None # Centroids across iterations
        self.labels_history_ = None  # Assignments across iterations
        self.sil_history_ = None     # Silhouette proxy across iterations
        self.weights_history_ = None # Weights across iterations

    def _initialization(self, X, n_clusters):
        X = np.asarray(X)
        if X.shape[0] < n_clusters:
           raise ValueError(f"n_clusters ({n_clusters}) can not exceed n_samples ({X.shape[0]}).")

        kmeans = KMeans(n_clusters=n_clusters,
                        init=self.init,
                        random_state=self.random_state,
                        n_init=1,
                        max_iter=1).fit(X)
        centers = kmeans.cluster_centers_
        labels = kmeans.labels_

        # Retry if some clusters are empty in the initial assignment
        if np.unique(labels).size < n_clusters:
           max_retries = 10 # 10 retries at max
           base_seed = self.random_state
           for attempt in range(1, max_retries + 1):
               seed = base_seed + attempt
               km = KMeans(n_clusters=n_clusters,
                           init=self.init,
                           random_state=seed,
                           n_init=1,
                           max_iter=1).fit(X)
               centers = km.cluster_centers_
               labels = km.labels_
               if np.unique(labels).size == n_clusters:
                  break
           else:
               raise ValueError(
                     f"KMeans (1-iter) initialization produced empty clusters after {max_retries} retries. "
                     f"Try a different random_state, or init='k-means++'.")
        return centers, labels

    def _fit_once(self, X, n_clusters, previous_centers, w):
        X = np.asarray(X)
        w = np.asarray(w)
        km = KMeans(
             n_clusters=n_clusters,
             init=previous_centers,
             n_init=1,
             max_iter=1,
             random_state=self.random_state)
        km.fit(X, sample_weight=w)

        centers = km.cluster_centers_
        labels = km.labels_
        return centers, labels

    @staticmethod
    def sil_scores(X, labels, centers):
        X = np.asarray(X, dtype=float)
        labels = np.asarray(labels, dtype=int)
        centers = np.asarray(centers, dtype=float)

        if len(X) == 0:
           raise ValueError("X is empty. Cannot compute silhouette scores.")

        n = X.shape[0]

        # Squared distances to all centroids
        D = euclidean_distances(X, centers, squared=True)

        # Distance to own centroid (a: intra cluster distance proxy)
        D_diag = D[np.arange(n), labels]  # a^2
        a_vals = np.sqrt(D_diag)  # a

        # Nearest other centroid distance (b: inter cluster distance)
        D_others = D.copy()
        D_others[np.arange(n), labels] = np.inf
        b_sq = D_others.min(axis=1)  # b^2
        b_vals = np.sqrt(b_sq)  # b

        # Silhouette surrogate (b-a)/[max{a,b}+epsilon]
        max_ab = np.maximum(np.maximum(a_vals, b_vals), 1e-12)
        s_vals = (b_vals - a_vals) / max_ab

        return s_vals, a_vals, b_vals

    @staticmethod
    def micro_sil(s_vals):
        s_vals = np.asarray(s_vals, dtype=float)
        if s_vals.size == 0:
           return 0.0
        return float(s_vals.mean())

    @staticmethod
    def macro_sil(s_vals, labels):
        s_vals = np.asarray(s_vals, dtype=float)
        labels = np.asarray(labels, dtype=int)

        if s_vals.size == 0 or labels.size == 0:
           return 0.0

        uniq, inv = np.unique(labels, return_inverse=True)
        if uniq.size == 0:
           return 0.0
        sum_per_cluster = np.bincount(inv, weights=s_vals)
        cnt_per_cluster = np.bincount(inv)

        valid = cnt_per_cluster > 0
        if not np.any(valid):
           return 0.0
        cluster_means = sum_per_cluster[valid] / cnt_per_cluster[valid]
        return float(cluster_means.mean())

    def _weights(self, s_vals, temperature):
        s_vals = np.asarray(s_vals, dtype=float)
        weights = np.exp(s_vals * temperature)
        return weights

    def _KSil(self, X, n_clusters, max_iter):

        # Initialize centroids
        centers, labels = self._initialization(X, n_clusters)
        tau = self.init_temperature # initial temperature
        prev_score =  None

        centers_history = [centers.copy()]
        labels_history = [labels.copy()]
        sil_history = []
        weights_history = []

        n_iter = 0

        while n_iter < max_iter:

            n_iter += 1

            # Compute point-silhouette scores
            sil_vals, a_vals, b_vals = KSil.sil_scores(X, labels, centers)
            score = KSil.macro_sil(sil_vals, labels)

            micro_sil = KSil.micro_sil(sil_vals)
            sil_history.append(micro_sil)

            # Update temperature
            if prev_score is not None:
               r = (score - prev_score) / max(1 - prev_score, 1e-12)

               r = float(np.clip(r, -1.0, 1.0))

               # data-driven bounds to avoid softmax saturation
               counts = np.bincount(labels, minlength=n_clusters)
               m_max = int(max(counts.max(), 2))
               L     = float(np.sqrt(2.0 * np.log(m_max)))
               z_max = (m_max - 1) / m_max
               tau_min, tau_max = 1e-12, L / max(z_max, 1e-8)

               # multiplicative update (no logs needed)
               eta = self.learning_rate # default: 0.2 (small stable learning rate)

               # Update temperature based on the rate of change of the silhouette objective
               tau = float(np.clip(tau * np.exp(eta * r), tau_min, tau_max))

            # store for next iteration’s drift calc
            prev_score = score

            # Retain previous centroids for convergence checking
            previous_centers = centers.copy()

            # Compute weights based on silhouette scores
            weights = self._weights(sil_vals, tau)

            weights_history.append(weights.copy())

            # Update centroids and assignments
            centers, labels = self._fit_once(X, n_clusters, previous_centers, weights)

            centers_history.append(centers.copy())
            labels_history.append(labels.copy())

            avg_move = np.linalg.norm(centers - previous_centers, axis=1).mean()

            # Centroid stability
            if avg_move < self.tol:
               break

        return centers, labels, n_iter, sil_history, centers_history, labels_history, weights_history

    def fit(self, X):
        """
        Run KSil on X and store the converged partition.
        X: array-like, shape (n_samples, n_features)
        """
        X_arr = X.values if hasattr(X, "values") else np.asarray(X, dtype=float)

        if self.n_init == 1:
           (centers,
           labels,
           n_iter,
           sil_history,
           centers_history,
           labels_history,
           weights_history) = self._KSil(X_arr, self.n_clusters, self.max_iter)
        else:
           base_seed = self.random_state
           seeds = [int(base_seed) + i for i in range(self.n_init)]

           # Helper: directly call _KSil with a seed
           def _run_one(seed):
               self.random_state = seed
               return self._KSil(X_arr, self.n_clusters, self.max_iter)

           if self.n_jobs in (None, 1):
              results = [_run_one(seed) for seed in seeds]
           else:
              results = Parallel(n_jobs=self.n_jobs)(delayed(_run_one)(seed) for seed in seeds)

           # Pick best run by final silhouette
           best_idx = None
           best_sil = -np.inf

           for i, res in enumerate(results):
               (_, _, _, sil_history_i,
               _, _, _) = res
               if sil_history_i:
                  final_sil_i = sil_history_i[-1]
               else:
                  final_sil_i = -np.inf
               if final_sil_i > best_sil:
                   best_sil = final_sil_i
                   best_idx = i

           (centers,
           labels,
           n_iter,
           sil_history,
           centers_history,
           labels_history,
           weights_history) = results[best_idx]

           self.random_state = base_seed # restore base random_state

        self.cluster_centers_ = centers
        self.labels_ = labels
        self.n_iter_ = n_iter

        self.sil_history_ = sil_history
        self.sil_ = sil_history[-1] if sil_history else None
        self.centers_history_ = centers_history
        self.labels_history_ = labels_history
        self.weights_history_ = weights_history

        return self

    def predict(self, X):
        """
        Assign new points to the nearest learned centroid.
        """
        if self.cluster_centers_ is None:
           raise ValueError("KSil model is not fitted yet. Call '.fit(...)' first.")

        X_arr = X.values if hasattr(X, "values") else np.asarray(X, dtype=float)
        dist_matrix = euclidean_distances(X_arr, self.cluster_centers_, squared=True)
        labels = np.argmin(dist_matrix, axis=1)
        return labels.astype(int)

    def transform(self, X):
        """
        Return distances of each sample to each centroid.
        """
        if self.cluster_centers_ is None:
           raise ValueError("KSil model is not fitted yet. Call '.fit(...)' first.")

        X_arr = X.values if hasattr(X, "values") else np.asarray(X, dtype=float)
        distances = euclidean_distances(X_arr, self.cluster_centers_, squared=False)
        return distances

    def fit_predict(self, X):
        """
        Equivalent to: fit(X); return labels_.
        """
        self.fit(X)
        return self.labels_

    def fit_transform(self, X):
        """
        Equivalent to: fit(X); return transform(X).
        """
        self.fit(X)
        return self.transform(X)

#### **Composite Silhouette**
We use Composite Silhouette (https://arxiv.org/abs/2604.13816) for cluster-count selection.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.utils import resample
import matplotlib.pyplot as plt
from joblib import Parallel, delayed


class CompSil:
    """
    Composite Silhouette (CompSil)

    Per subsample b=1..B for each k:
      - compute S_micro^(b), S_macro^(b)
      - Δ_b = S_micro^(b) - S_macro^(b)
      - D = max_b |Δ_b|
      - r_raw_b = Δ_b/(D+eps) in [-1,1]
      - r_b = tanh(r_raw_b)
      - w_b = (1+r_b)/2
      - S_mM^(b) = w_b*S_micro^(b) + (1-w_b)*S_macro^(b)

    Aggregate for each k:
      - mean score:   S_mM(k)  = mean_b S_mM^(b)
      - std score:    std_S_mM(k) = std_b  S_mM^(b)
      - standard error: se_S_mM(k) = std_S_mM(k) / sqrt(B_eff)
      - lower conf bound: LCB_S_mM(k) = S_mM(k) - se_S_mM(k)

    Selection:
      - k* = argmax_k S_mM(k) (or argmax_k LCB_S_mM(k))

    Sampling size mechanism:
      - explicit `sample_size` (int), or
      - sample_size=None or sample_size="auto", the class chooses a subsample size automatically
        based on dataset size N and the maximum k in `k_values`.

    Parameters:
    - data: ndarray or DataFrame, shape (n_samples, n_features)
    - ground_truth: int, optional
    - k_values: iterable or int, default=range(2, 11)
    - num_samples: int, default=100  (B)
    - sample_size: int | float | None | "auto", default=1000
        * int: absolute subsample size m
        * float in (0,1]: treated as fraction f, m = floor(f*N)
        * None or "auto": compute m automatically (see _auto_sample_size)
    - random_state: int, default=42
    - n_jobs: int, default=-1
    - eps: float, default=1e-12
    """

    def __init__(self,
                 data,
                 ground_truth=None,
                 k_values=range(2, 11),
                 num_samples=10,
                 sample_size="auto",
                 random_state=42,
                 n_jobs=-1,
                 eps=1e-12):
        self.data = data
        self.ground_truth = ground_truth
        self.k_values = [k_values] if isinstance(k_values, int) else list(k_values)
        self.num_samples = int(num_samples)
        self.random_state = int(random_state)
        self.n_jobs = int(n_jobs)
        self.eps = float(eps)

        self._results = []
        self.results_df = pd.DataFrame()
        self.score_ = None  # set only if one k is evaluated

        self.n_samples_ = int(len(self.data))
        if self.n_samples_ <= 0:
            raise ValueError("Empty dataset.")

        self.sample_size = self._resolve_sample_size(sample_size)
        self.sample_fraction_ = self.sample_size / self.n_samples_

        if self.sample_size < 2:
            raise ValueError(f"Resolved sample_size={self.sample_size} is too small.")
        if self.sample_size > self.n_samples_:
            raise ValueError(
                f"Resolved sample_size={self.sample_size} is larger than n_samples={self.n_samples_}."
            )

    def _resolve_sample_size(self, sample_size):
        n = self.n_samples_

        # Auto
        if sample_size is None or (isinstance(sample_size, str) and sample_size.lower() == "auto"):
            return self._auto_sample_size()

        # Fraction mode (float in (0,1])
        if isinstance(sample_size, float):
            if not (0.0 < sample_size <= 1.0):
                raise ValueError("If sample_size is a float, it must be in (0, 1].")
            m = int(np.floor(sample_size * n))
            return max(2, min(m, n))

        # Int mode
        m = int(sample_size)
        return m

    def _auto_sample_size(self):
        """
        Automatic subsample size selection (no user-facing hyperparameters).

        Heuristic:
          1) Ensure a minimum average points-per-cluster at k_max: m >= 30 * k_max
          2) Use a baseline fraction depending on dataset size:
             - small N: 0.8N
             - medium N: 0.6N
             - large N: 0.4N
          3) Take the maximum of (1) and (2), then cap at N.
        """
        n = self.n_samples_
        k_max = int(max(self.k_values)) if len(self.k_values) > 0 else 2

        m_min = 30 * k_max

        if n <= 2000:
            m_base = int(np.floor(0.80 * n))
        elif n <= 20000:
            m_base = int(np.floor(0.60 * n))
        else:
            m_base = int(np.floor(0.40 * n))

        m = max(m_min, m_base)
        m = min(max(2, m), n)
        return m

    def evaluate_sample(self, k, i):
        """
        One subsampling iteration for fixed k.
        Returns: (smicro, smacro, diff, s_mm_b)
        """
        seed = self.random_state + i

        sampled_data = resample(
            self.data,
            n_samples=self.sample_size,
            replace=False,
            random_state=seed
        )

        kmeans = KMeans(n_clusters=k, random_state=seed, n_init=1)
        labels = kmeans.fit_predict(sampled_data)

        try:
            s = silhouette_samples(sampled_data, labels)  # compute once

            # micro silhouette
            smicro = float(np.mean(s))

            labs = np.asarray(labels)
            uniq = np.unique(labs)
            cluster_means = [float(np.mean(s[labs == u])) for u in uniq]

            # macro silhouette
            smacro = float(np.mean(cluster_means)) if len(cluster_means) > 0 else np.nan
        except Exception:
            return np.nan, np.nan, np.nan, np.nan

        diff = smicro - smacro
        return smicro, smacro, diff, np.nan

    @staticmethod
    def _tanh_rb_weights_from_differences(differences, eps=1e-12):
        """
        Given Δ_b over b=1..B, compute:
          D       = max |Δ_b|
          r_raw_b = Δ_b/(D+eps) in [-1, 1]
          r_b     = tanh(r_raw_b)
          w_b     = (1+r_b)/2 in (0,1)
        """
        d = np.asarray(differences, dtype=float)
        finite = np.isfinite(d)

        if not np.any(finite):
            return d * np.nan, d * np.nan, np.nan

        D = float(np.max(np.abs(d[finite])))
        denom = D + float(eps)

        if D == 0.0:
            r = np.zeros_like(d)
            r[~finite] = np.nan
            w = 0.5 * np.ones_like(d)
            w[~finite] = np.nan
            return w, r, D

        r_raw = d / denom
        r_raw = np.clip(r_raw, -1.0, 1.0)
        r_raw[~finite] = np.nan

        r = np.tanh(r_raw)
        r[~finite] = np.nan

        w = 0.5 * (1.0 + r)
        w[~finite] = np.nan

        return w, r, D

    def evaluate(self):
        """
        Evaluate over k_values using subsampled clustering.
        Stores results in self.results_df.

        Output columns (per k):
        - avg S_micro
        - avg S_macro
        - w_micro (mean of per-subsample weights; descriptive)
        - S_mM (mean of per-subsample composites)
        - std S_mM
        - se S_mM
        - LCB S_mM  (S_mM - se)
        """
        self._results = []

        for k in self.k_values:
            results = Parallel(n_jobs=self.n_jobs)(
                delayed(self.evaluate_sample)(k, i) for i in range(self.num_samples)
            )

            smicro_list, smacro_list, differences, _ = zip(*results)

            smicro_arr = np.asarray(smicro_list, dtype=float)
            smacro_arr = np.asarray(smacro_list, dtype=float)
            diff_arr = np.asarray(differences, dtype=float)

            avg_smicro = float(np.nanmean(smicro_arr)) if np.any(np.isfinite(smicro_arr)) else np.nan
            avg_smacro = float(np.nanmean(smacro_arr)) if np.any(np.isfinite(smacro_arr)) else np.nan

            # weights
            w_b, r_b, D = self._tanh_rb_weights_from_differences(diff_arr, eps=self.eps)

            # per-subsample composite
            S_b = w_b * smicro_arr + (1.0 - w_b) * smacro_arr

            # mean composite
            S_mM = float(np.nanmean(S_b)) if np.any(np.isfinite(S_b)) else np.nan

            # descriptive mean weight
            w_micro_mean = float(np.nanmean(w_b)) if np.any(np.isfinite(w_b)) else np.nan

            # LCB components computed from S_b across valid subsamples
            finite_sb = np.isfinite(S_b)
            B_eff = int(np.sum(finite_sb))
            if B_eff >= 2:
                std_smm = float(np.nanstd(S_b, ddof=1))
                se_smm = std_smm / np.sqrt(B_eff)
            elif B_eff == 1:
                std_smm = 0.0
                se_smm = 0.0
            else:
                std_smm = np.nan
                se_smm = np.nan

            lcb_smm = (S_mM - se_smm) if (np.isfinite(S_mM) and np.isfinite(se_smm)) else np.nan

            if len(self.k_values) == 1:
                self.score_ = S_mM

            result = {
                'k': int(k),
                'avg S_micro': avg_smicro,
                'avg S_macro': avg_smacro,
                'w_micro': w_micro_mean,
                'S_mM': S_mM,
                'std S_mM': std_smm,
                'se S_mM': se_smm,
                'LCB S_mM': lcb_smm,
                'B_eff': B_eff,
                'sample_size': int(self.sample_size),
                'sample_fraction': float(self.sample_fraction_),
            }
            self._results.append(result)

        self.results_df = pd.DataFrame(self._results)

    def plot_results(self):
        """
        Plot S_mM and individual averages vs k.
        """
        if self.results_df.empty:
            raise ValueError("No results available. Run evaluate() first.")
        if len(self.results_df) == 1:
            raise ValueError("Cannot plot with only one k. Evaluate multiple k values.")

        max_smicro = self.results_df['avg S_micro'].max()
        max_smicro_k = self.results_df.loc[self.results_df['avg S_micro'].idxmax(), 'k']

        max_smacro = self.results_df['avg S_macro'].max()
        max_smacro_k = self.results_df.loc[self.results_df['avg S_macro'].idxmax(), 'k']

        max_smm = self.results_df['S_mM'].max()
        max_smm_k = self.results_df.loc[self.results_df['S_mM'].idxmax(), 'k']

        plt.figure(figsize=(10, 4))

        if self.ground_truth is not None:
            plt.axvline(
                x=self.ground_truth, color='red', linestyle='--', linewidth=2.5,
                label='Ground Truth'
            )

        plt.plot(
            self.results_df['k'], self.results_df['avg S_micro'],
            marker='o', linestyle='-', color='orange', linewidth=4, markersize=8, label='avg S_micro'
        )
        plt.plot(max_smicro_k, max_smicro, marker='*', color='orange', markersize=18)

        plt.plot(
            self.results_df['k'], self.results_df['avg S_macro'],
            marker='o', linestyle='-', color='blue', linewidth=4, markersize=8, label='avg S_macro'
        )
        plt.plot(max_smacro_k, max_smacro, marker='*', color='blue', markersize=18)

        plt.plot(
            self.results_df['k'], self.results_df['S_mM'],
            marker='o', linestyle='--', color='green', linewidth=4, markersize=8, label='S_mM'
        )
        plt.plot(max_smm_k, max_smm, marker='*', color='green', markersize=18)

        plt.xlabel('k', fontsize=15)
        plt.xticks(self.k_values, fontsize=14)
        plt.yticks(fontsize=14)
        plt.tick_params(axis='y', which='both', length=0)
        plt.grid(axis='y', linestyle='--')

        ax = plt.gca()
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_visible(False)
        ax.spines['top'].set_visible(False)

        handles, labels = ax.get_legend_handles_labels()
        ax.legend(
            handles, labels,
            loc="lower left",
            bbox_to_anchor=(0, 1.02, 1, 0.2),
            mode="expand",
            ncol=len(labels),
            frameon=False,
            fontsize=12
        )

        plt.tight_layout()
        plt.show()

    def get_optimal_k(self, use_lcb=False):
        """
        Return optimal k.

        By default, uses max selection:
          k* = argmax_k (S_mM) (use_lcb for argmax_k (LCB S_mM) selection)

        """
        if self.results_df.empty:
            raise ValueError("No results available. Run evaluate() first.")

        if len(self.results_df) == 1:
            return int(self.results_df['k'].iloc[0])

        col = 'LCB S_mM' if use_lcb else 'S_mM'
        if col not in self.results_df.columns:
            raise ValueError(f"Missing column '{col}'. Run evaluate() first.")

        optimal_row = self.results_df.loc[self.results_df[col].idxmax()]
        return int(optimal_row['k'])

    def get_results_dataframe(self):
        """
        Return results DataFrame indexed by k.
        """
        if self.results_df.empty:
            raise ValueError("No results available. Run evaluate() first.")
        return self.results_df.set_index('k', inplace=False)

#### **CAKE**
We have used CAKE (https://arxiv.org/abs/2602.18435) previously for confidence estimation when clustering the proceedings paragraphs. Currently we use some methods of the class for fast (approximate) Silhouette computation.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.metrics import silhouette_samples, confusion_matrix
from scipy.optimize import linear_sum_assignment
from sklearn.preprocessing import LabelEncoder

def sil_samples(X, labels, approximation=False, centers=None):
    """
    Compute silhouette scores for each point in the dataset,
    with approximate fast centroid-based computation option.
    """
    # Ensure arrays
    X = np.asarray(X)
    labels = np.asarray(labels)
    if X.ndim != 2:
       raise ValueError("X must be a 2D array of shape (n_samples, n_features).")
    if labels.ndim != 1 or labels.shape[0] != X.shape[0]:
       raise ValueError("labels must be a 1D array of length n_samples.")

    unique_labels, inv = np.unique(labels, return_inverse=True)
    k = unique_labels.size
    if k < 2:
       raise ValueError("Silhouette computation requires at least 2 clusters.")

    # Exact silhouette scores
    if approximation == False:
       silhouette_scores = silhouette_samples(X, labels=labels)
       return silhouette_scores

    # Centroid-based approximate silhouette scores
    n_samples, n_features = X.shape

    if centers is None:
       centers = np.array([X[inv == i].mean(axis=0) for i in range(k)], dtype=float)
    else:
       centers = np.asarray(centers, dtype=float)
       if centers.ndim != 2 or centers.shape[1] != n_features:
          raise ValueError(f"centers must have shape (k, d) with d={n_features}.")
       if centers.shape[0] != k:
          raise ValueError(f"centers.shape[0] must equal number of clusters k={k}.")
       if not np.array_equal(unique_labels, np.arange(k)):
          raise ValueError("When passing ndarray centers, labels must be dense 0..k-1.")

    # Squared distances to all centroids
    D_sq = euclidean_distances(X, centers, squared=True)

    # a(i): distance to own centroid
    a = np.sqrt(np.maximum(D_sq[np.arange(n_samples), inv], 0.0))

    # b(i): distance to nearest other centroid
    D_sq[np.arange(n_samples), inv] = np.inf
    b = np.sqrt(np.min(D_sq, axis=1))

    # Silhouette per point
    denom = np.maximum(np.maximum(a, b), 1e-12)
    s_point = (b - a) / denom

    # Singleton clusters -> silhouette = 0
    counts = np.bincount(inv, minlength=k).astype(int)
    s_point[counts[inv] < 2] = 0.0

    silhouette_scores = np.clip(s_point, -1.0, 1.0)

    return silhouette_scores

def sil_samples_stats(X, labels_list, approximation=False, centers_list=None):
    """
    Compute and aggregate silhouette scores across multiple clustering runs,
    returning per-sample mean and standard deviation of silhouette scores.
    """
    X = np.asarray(X)
    n_samples = X.shape[0]
    n_runs = len(labels_list)
    if n_runs < 2:
       raise ValueError("Clustering Ensemble must contain at least 2 partitions.")

    if centers_list is not None and len(centers_list) != n_runs:
       raise ValueError("centers_list must match the number of labelings in labels_list")

    # Initialize matrix to hold silhouette scores
    sil_scores = np.zeros((n_runs, n_samples), dtype=float)

    for i, labels in enumerate(labels_list):
        labels_arr = np.asarray(labels)
        centers = None
        if approximation and centers_list is not None:
           centers = centers_list[i]

        # Compute silhouette scores for this run
        sil_scores[i] = sil_samples(X, labels_arr, approximation=approximation, centers=centers)

    # Compute mean and standard deviation of sample-silhouette scores over the ensemble
    mean_sil_samples = sil_scores.mean(axis=0)
    std_sil_samples = sil_scores.std(axis=0)

    return mean_sil_samples, std_sil_samples

def align_labels(target, source):
    """
    Aligns the labels in "source" to match the labels in "target"
    using the Hungarian Algorithm based on a contingency matrix.

    Parameters:
    - target: array-like of shape (n_samples,)
              The reference label vector to align to.
    - source: array-like of shape (n_samples,)
              The label vector to be permuted for alignment.

    Returns:
    - aligned: np.ndarray of shape (n_samples,)
               The source labels, remapped to best match the target labels.
    """
    target = np.asarray(target)
    source = np.asarray(source)
    unique_target = np.unique(target)
    unique_source = np.unique(source)
    if len(unique_target) != len(unique_source):
       raise ValueError(
           f"Cannot align: Target has {len(unique_target)} clusters {unique_target.tolist()}, "
           f"but source has {len(unique_source)} clusters {unique_source.tolist()}"
       )
    # Encode labels to indices based on their unique values
    le_target = LabelEncoder().fit(unique_target)
    le_source = LabelEncoder().fit(unique_source)

    target_encoded = le_target.transform(target)
    source_encoded = le_source.transform(source)

    # Confusion matrix with indices aligned to encoded labels
    c_matrix = confusion_matrix(target_encoded, source_encoded)

    # Apply the Hungarian algorithm
    row_ind, col_ind = linear_sum_assignment(-c_matrix)

    # Create mapping from source labels to target labels
    mapping = {
        le_source.classes_[src_col]: le_target.classes_[tgt_row]
        for tgt_row, src_col in zip(row_ind, col_ind)
    }
    # Remap source labels using the mapping
    aligned = np.vectorize(mapping.get)(source)

    return aligned

def pairwise_stability(labels_runs):
    """
    Computes per-point clustering stability across multiple runs by aligning labels
    pairwise using the Hungarian algorithm to account for label permutations.

    Parameters:
    - labels_runs: list of array-like
        List of label arrays from multiple clustering runs. Each array must have the
        same number of samples and clusters (unique labels).

    Returns:
    - stability: np.ndarray of shape (n_samples,)
        Per-point stability score between 0 and 1, indicating the fraction of run-pairs
        where the point's label matches after optimal alignment.

    Notes:
    - Requires all runs to have the same number of clusters (unique labels).
    - Label alignment is done pairwise between runs using the Hungarian algorithm.
    - High stability (~1) indicates stable cluster assignments across runs.
    """
    labels_runs = [np.asarray(labels) for labels in labels_runs]
    n_runs, n_samples = len(labels_runs), len(labels_runs[0])
    if n_runs < 2:
       raise ValueError("pairwise_stability needs at least 2 runs.")

    # Counters: How many run-pairs agree per point
    agreement_counts = np.zeros(n_samples, dtype=int)
    total_pairs = 0

    # For every unique pair of runs
    for r1 in range(n_runs):
        labels_r1 = labels_runs[r1]
        for r2 in range(r1+1, n_runs):
            labels_r2 = labels_runs[r2]

            # Align labels_r2 to labels_r1 using Hungarian
            aligned_r2 = align_labels(labels_r1, labels_r2)
            matches = (labels_r1 == aligned_r2)

            # Add matches to total per-point agreement count
            agreement_counts += matches
            total_pairs += 1

    return agreement_counts/total_pairs

def cake(X, labels_list, method='product', approximation=False, centers_list=None, geom_norm='clip'):
    """
    Compute a confidence score per point for clustering ensembles, defined as:
    stability_i * geometric_stability_i or
    2 * stability_i * geometric_stability_i / [stability_i + geometric_stability_i]
    where stability is the pairwise label agreement across runs,
    and mean_sil_i, std_sil_i are the statistics from sil_samples_stats.

    Parameters:
    - X: array-like, shape (n_samples, n_features)
    - labels_list: list of array-like, each of shape (n_samples,)
        A list of cluster labelings for the same dataset X.
    - approximation: bool, default=False
        Whether to use the approximate silhouette computation.
    - centers_list: list of pd.Series or array-like, optional
        If approximation=True, an optional list of centroids for each clustering.
    - method: str, default='product'
        'product' or 'harmonic_mean' for the formulation of the CAKE scores.
    - geo_norm: str, default='clip'
        - 'affine' (default): maps geom_raw in [-1,1] to [0,1] via (x+1)/2 (preserves negatives).
        - 'clip': max(mean_sil - std_sil, 0) clipped at 1 (original behavior; discards negatives).

    Returns:
    - cake_scores: np.ndarray, shape (n_samples,)
        The CAKE scores for each point.
    - stability: np.ndarray, shape (n_samples,)
        The stability scores for each point.
    - geom_stability: np.ndarray, shape (n_samples,)
        The silhouette-based reliability scores for each point (mean_sil_i - std_sil_i).
    - summary: pd.DataFrame
        Summary of above metrics per point in X.
    """
    # Compute silhouette statistics
    mean_sil, std_sil = sil_samples_stats(X, labels_list,
                                          approximation=approximation,
                                          centers_list=centers_list)

    # Assignment stability across the ensemble
    stability = pairwise_stability(labels_list)

    # Silhouette-based stability across the ensemble
    geom_raw = mean_sil - std_sil
    if geom_norm == 'clip':
       geom_stability = np.clip(mean_sil - std_sil, 0.0, 1.0)
    else:
       geom_stability = np.clip((geom_raw + 1) / 2, 0, 1) # preserves information for negative scores

    # Calculate confidence scores
    if method == 'product':
       cake_scores = stability * geom_stability
    elif method == 'harmonic_mean':
       num = 2.0 * stability * geom_stability
       denom = np.maximum(stability + geom_stability, 1e-8)
       cake_scores = num / denom
    else:
       raise ValueError(
           f"Unknown method: {method}. Use 'product' or 'harmonic_mean'."
       )

    # Summary DataFrame
    summary = pd.DataFrame({
        'Mean Silhouette': mean_sil,
        'STD Silhouette': std_sil,
        'Geometric Stability': geom_stability,
        'Assignment Stability': stability,
        'CAKE': cake_scores
    })

    return cake_scores, stability, geom_stability, summary



import numpy as np
import pandas as pd
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from collections import OrderedDict
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.metrics import adjusted_rand_score
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap, Normalize

from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    silhouette_samples,
    adjusted_rand_score,
    normalized_mutual_info_score
)
from scipy.stats import spearmanr
from scipy.stats import t
from sklearn.metrics import f1_score, adjusted_mutual_info_score
from scipy.stats import kendalltau, spearmanr
from sklearn.mixture import GaussianMixture
from sklearn.metrics import average_precision_score, roc_auc_score

from sklearn.decomposition import PCA
from collections import Counter
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import NearestNeighbors
import time, gc

def clustering_accuracy(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    row_ind, col_ind = linear_sum_assignment(-cm)
    accuracy = cm[row_ind, col_ind].sum() / np.sum(cm)
    return accuracy

def micro_sil(X, labels, approximation=False, centers=None):
    s = sil_samples(X, labels, approximation=approximation, centers=centers)
    return float(np.mean(s))

def macro_sil(X, labels, approximation=False, centers=None):
    labels = np.asarray(labels)
    s = sil_samples(X, labels, approximation=approximation, centers=centers)
    uniq = np.unique(labels)
    cluster_means = [s[labels == lab].mean() for lab in uniq]
    return float(np.mean(cluster_means))

def consensus_medoid_majority(labels_runs):
    if not labels_runs:
        raise ValueError("labels_runs must be a non-empty list of 1D arrays.")
    labs = [np.asarray(l) for l in labels_runs]
    R = len(labs)
    n = labs[0].shape[0]
    if any(len(l) != n for l in labs):
        raise ValueError("All runs must have the same number of samples.")

    ks = {np.unique(l).size for l in labs}
    if len(ks) != 1:
        raise ValueError(f"All runs must have the same number of clusters; got {sorted(ks)}.")

    # pairwise symmetric Hamming distances after alignment
    D = np.zeros((R, R), dtype=float)
    for i in range(R):
        li = labs[i]
        for j in range(i+1, R):
            lj = labs[j]
            aligned_j = align_labels(li, lj)
            d_ij = 1.0 - np.mean(aligned_j == li)
            aligned_i = align_labels(lj, li)
            d_ji = 1.0 - np.mean(aligned_i == lj)
            D[i, j] = D[j, i] = 0.5 * (d_ij + d_ji)

    medoid = int(np.argmin(D.sum(axis=1)))
    ref = labs[medoid]
    aligned = [align_labels(ref, l) for l in labs]   # align all runs to the medoid
    A = np.vstack(aligned)                           # (R, n)

    # majority vote per point; tie -> medoid label
    z_star = np.empty(n, dtype=ref.dtype)
    for i in range(n):
        counts = Counter(A[:, i])
        top = counts.most_common(2)
        if len(top) == 1 or top[0][1] > top[1][1]:
            z_star[i] = top[0][0]
        else:
            z_star[i] = ref[i]  # tie-break to medoid label

    agree = np.mean(A == z_star, axis=0)
    return z_star, agree

### Load Narrative Embeddings

In [ ]:
import numpy as np

file_path = './out_files/narrative_embeddings.npy'

try:
    X = np.load(file_path)
    print("loaded")
    print(f"\nEmbeddings (Samples, Dimensions): {X.shape}")
except FileNotFoundError:
    print(f"File not found: {file_path}")

### Load the Narrative Dataset

In [ ]:
import pandas as pd

file_path = './out_files/06_tempi_df_R1_final_narratives.csv'

try:
    df = pd.read_csv(file_path)
    print(f"Shape: {df.shape}")
    display(df.head(3))
except FileNotFoundError:
    print(f"File not found: {file_path}")

PCA on Embeddings

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=200, random_state=0)
X_pca = pca.fit_transform(X_scaled)

print(f"New shape: {X_pca.shape}")

### Select the Optimal Number of Narrative Clusters $k$ with Composite Silhouette

In [ ]:
compsilhouette = CompSil(
    data=X_pca,
    ground_truth=None,
    k_values=range(5,30),
    num_samples=20,
    sample_size="auto",
    random_state=0
)

compsilhouette.evaluate()

df_X= compsilhouette.get_results_dataframe()
display(df_X)

compsilhouette.plot_results()

In [ ]:
k_smm = compsilhouette.get_optimal_k(use_lcb=False)

### Cluster the Narrative Embeddings with $k$-means and K-Sil



To partition the narrative embeddings into interpretable groups, we apply two clustering approaches using the same number of clusters, $(k = k_{S_{mM}})$, selected by Composite Silhouette ($S_{mM}$) in the previous step:

- $k$-means, which assigns each narrative to the nearest centroid by minimizing within-cluster variance.
- **K-Sil**, a silhouette-driven instance-weighted $k$-means variant that reweights points at each iteration using a centroid-margin proxy for the silhouette score, thereby emphasizing confidently assigned instances and down-weighting ambiguous or noisy boundary points. Centroid updates are computed as softmax-weighted means, with an adaptive temperature that calibrates the sharpness of the weighting scheme using a cluster-balanced, macro-averaged silhouette criterion.

For each method, we:

1. assign a cluster label to every narrative,
2. compute a per-narrative silhouette-based cohesion score,
3. summarize each cluster by:
   - its **size**,
   - its **share** in the corpus,
   - its **mean silhouette score**,
4. visualize:
   - the **relative prevalence** of clusters,
   - the **average geometric cohesion** of each cluster.

The comparison allows us to assess not only how the narratives are distributed across clusters, but also how clearly separated those clusters are in embedding space.


**Model choice:**


We retain the **K-Sil solution** for the downstream analysis because it produces **higher silhouette values**, indicating more coherent and more distinct narrative clusters.

**Next step: diagnosing cluster overlap**


Even when silhouette values are relatively high, some clusters may still be substantively close in meaning.  
For that reason, after selecting the K-Sil solution, we conduct a **cluster-overlap diagnosis** to identify:

- which clusters are nearest to one another,
- which narrative themes are shared across adjacent clusters,
- and which terms or examples make one cluster substantively different from another.

$k$-means Clustering:

In [ ]:
n_clusters = k_smm

kmeans = KMeans(n_clusters=n_clusters, max_iter=100, random_state=0, n_init=100)
cluster_labels = kmeans.fit_predict(X_pca)

new_df_kmeans = df.copy()
new_df_kmeans['narratives_label'] = cluster_labels

sil_vals, _, _ = KSil.sil_scores(X_pca, cluster_labels, kmeans.cluster_centers_)
new_df_kmeans['ksil_score'] = sil_vals

display(new_df_kmeans.head(5))

cluster_profiles_kmeans = new_df_kmeans.groupby('narratives_label').agg(
    Size=('narrative', 'count'),
    mean_Silhouette=('ksil_score', 'mean')
).reset_index()

cluster_profiles_kmeans['share'] = (cluster_profiles_kmeans['Size'] / len(new_df_kmeans)) * 100

final_report_kmeans = cluster_profiles_kmeans.rename(columns={'narratives_label': 'Cluster ID'}).set_index('Cluster ID')
display(final_report_kmeans[['Size', 'share', 'mean_Silhouette']])

plt.figure(figsize=(16, 6))
ax1 = sns.barplot(x=cluster_profiles_kmeans['narratives_label'], y=cluster_profiles_kmeans['share'],
                  palette='magma', edgecolor='black')

for p in ax1.patches:
    ax1.annotate(f'{p.get_height():.1f}%',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=9, fontweight='bold',
                xytext=(0, 5), textcoords='offset points')

plt.title(f'Cluster Topic Share (K={n_clusters})', fontsize=15, pad=20)
plt.xlabel('Cluster ID', fontsize=12)
plt.ylabel('Percentage (%)', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(16, 6))
colors = sns.color_palette("coolwarm", len(cluster_profiles_kmeans))
ax2 = sns.barplot(x=cluster_profiles_kmeans['narratives_label'], y=cluster_profiles_kmeans['mean_Silhouette'],
                  palette=colors, edgecolor='black')

for p in ax2.patches:
    ax2.annotate(f'{p.get_height():.3f}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=9, fontweight='bold',
                xytext=(0, 5), textcoords='offset points')

global_mean = cluster_profiles_kmeans['mean_Silhouette'].mean()
plt.axhline(y=global_mean, color='red', linestyle='--',
            label=f"Global Mean: {global_mean:.4f}")

plt.title(f'Cluster Geometric Cohesion (Mean Silhouette)', fontsize=15, pad=20)
plt.xlabel('Cluster ID', fontsize=12)
plt.ylabel('Silhouette Score', fontsize=12)
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
top_5_raw_kmeans = new_df_kmeans.sort_values('ksil_score', ascending=False).groupby('narratives_label').head(5).reset_index(drop=True)

def format_top_speeches(series):
    formatted = [f"{i+1}. {text}" for i, text in enumerate(series)]
    return " \n\n ".join(formatted)

collapsed_top_df_kmeans = top_5_raw_kmeans.groupby('narratives_label')['narrative'].apply(format_top_speeches).reset_index()

mean_sil_top = top_5_raw_kmeans.groupby('narratives_label')['ksil_score'].mean().reset_index()
collapsed_top_df_kmeans['mean_top_silhouette'] = mean_sil_top['ksil_score']

pd.set_option('display.max_colwidth', None)

display(collapsed_top_df_kmeans)

K-Sil Clustering:

In [ ]:
n_clusters = k_smm

ksil = KSil(n_clusters=n_clusters,
            max_iter=100,
            random_state=0,
            init_temperature=2.0,
            learning_rate=0.2,
            tol=1e-4,
            n_init=100,
            n_jobs=-1)

cluster_labels = ksil.fit_predict(X_pca)

new_df = df.copy()
new_df['narratives_label'] = cluster_labels

sil_vals, _, _ = KSil.sil_scores(X_pca, cluster_labels, ksil.cluster_centers_)
new_df['ksil_score'] = sil_vals

display(new_df.head(5))

cluster_profiles = new_df.groupby('narratives_label').agg(
    Size=('narrative', 'count'),
    mean_Silhouette=('ksil_score', 'mean')
).reset_index()

cluster_profiles['share'] = (cluster_profiles['Size'] / len(new_df)) * 100

final_report = cluster_profiles.rename(columns={'narratives_label': 'Cluster ID'}).set_index('Cluster ID')
display(final_report[['Size', 'share', 'mean_Silhouette']])

plt.figure(figsize=(16, 6))
ax1 = sns.barplot(x=cluster_profiles['narratives_label'], y=cluster_profiles['share'],
                  palette='magma', edgecolor='black')

for p in ax1.patches:
    ax1.annotate(f'{p.get_height():.1f}%',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=9, fontweight='bold',
                xytext=(0, 5), textcoords='offset points')

plt.title(f'Cluster Topic Share (K={n_clusters})', fontsize=15, pad=20)
plt.xlabel('Cluster ID', fontsize=12)
plt.ylabel('Percentage (%)', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(16, 6))
colors = sns.color_palette("coolwarm", len(cluster_profiles))
ax2 = sns.barplot(x=cluster_profiles['narratives_label'], y=cluster_profiles['mean_Silhouette'],
                  palette=colors, edgecolor='black')

for p in ax2.patches:
    ax2.annotate(f'{p.get_height():.3f}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=9, fontweight='bold',
                xytext=(0, 5), textcoords='offset points')

global_mean = cluster_profiles['mean_Silhouette'].mean()
plt.axhline(y=global_mean, color='red', linestyle='--',
            label=f"Global Mean: {global_mean:.4f}")

plt.title(f'Cluster Geometric Cohesion (Mean Silhouette)', fontsize=15, pad=20)
plt.xlabel('Cluster ID', fontsize=12)
plt.ylabel('Silhouette Score', fontsize=12)
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
top_5_raw = new_df.sort_values('ksil_score', ascending=False).groupby('narratives_label').head(5).reset_index(drop=True)

def format_top_speeches(series):
    formatted = [f"{i+1}. {text}" for i, text in enumerate(series)]
    return " \n\n ".join(formatted)

collapsed_top_df = top_5_raw.groupby('narratives_label')['narrative'].apply(format_top_speeches).reset_index()

mean_sil_top = top_5_raw.groupby('narratives_label')['ksil_score'].mean().reset_index()
collapsed_top_df['mean_top_silhouette'] = mean_sil_top['ksil_score']


pd.set_option('display.max_colwidth', None)

display(collapsed_top_df)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

plot_df = cluster_profiles.copy().sort_values("narratives_label").reset_index(drop=True)
plot_df["Cluster"] = plot_df["narratives_label"].apply(lambda x: f"Cluster {x}")

share_color = "#0048B1"
sil_color = "#D3212D"

x = np.arange(len(plot_df))

bar_width = 0.30
offset = 0.22 # left- right

fig, ax1 = plt.subplots(figsize=(13.2, 7.2))
ax2 = ax1.twinx()

bars1 = ax1.bar(
    x - offset,
    plot_df["share"],
    width=bar_width,
    color=share_color,
    edgecolor="black",
    linewidth=0.9,
    label="Corpus share (%)",
    zorder=3
)

bars2 = ax2.bar(
    x + offset,
    plot_df["mean_Silhouette"],
    width=bar_width,
    color=sil_color,
    edgecolor="black",
    linewidth=0.9,
    label="Mean silhouette",
    zorder=3
)

ax1.set_title(
    "",
    fontsize=18,
    fontweight="bold",
    pad=16
)

ax1.set_xticks(x)
ax1.set_xticklabels(plot_df["Cluster"], fontsize=13, fontweight="bold")
ax1.tick_params(axis="x", length=0)

# no axes
ax1.set_xlabel("")
ax1.set_ylabel("")
ax2.set_ylabel("")

ax1.tick_params(axis="y", left=False, labelleft=False)
ax2.tick_params(axis="y", right=False, labelright=False)

for spine in ["top", "right", "left", "bottom"]:
    ax1.spines[spine].set_visible(False)
    ax2.spines[spine].set_visible(False)

share_max = plot_df["share"].max()
sil_max = plot_df["mean_Silhouette"].max()

ax1.set_ylim(0, share_max + 7.5)
ax2.set_ylim(0, sil_max + 0.10)
ax1.grid(axis="y", linestyle="--", linewidth=0.7, alpha=0.22, zorder=0)
ax1.set_axisbelow(True)

share_offset = 0.85
sil_offset = 0.008

for bar in bars1:
    h = bar.get_height()
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        h + share_offset,
        f"{h:.1f}%",
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold",
        color=share_color,
        clip_on=False
    )

for bar in bars2:
    h = bar.get_height()
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        h + sil_offset,
        f"{h:.3f}",
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold",
        color=sil_color,
        clip_on=False
    )

handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    handles1 + handles2,
    labels1 + labels2,
    loc="upper left",
    bbox_to_anchor=(0.01, 1.03),
    ncol=2,
    frameon=False,
    fontsize=13
)

plt.tight_layout()

plt.show()

**We keep the higher silhouette (more distinct) K-Sil results and we perform a cluster overlap diagnosis.**

### Cluster-Overlap Diagnosis

In [ ]:
text_col = 'narrative'
label_col = 'narratives_label'

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

# Cluster centroids in PCA space
centroids = (
    new_df
    .groupby(label_col)
    .apply(lambda g: np.mean(X_pca[g.index], axis=0))
)

centroid_matrix = np.vstack(centroids.values)
cluster_ids = centroids.index.to_list()

# Cosine similarity between centroids
sim_matrix = cosine_similarity(centroid_matrix)
sim_df = pd.DataFrame(sim_matrix, index=cluster_ids, columns=cluster_ids)

# Remove self-similarity when searching nearest neighbor
np.fill_diagonal(sim_matrix, -np.inf)

nearest_cluster = sim_matrix.argmax(axis=1)
nearest_similarity = sim_matrix.max(axis=1)

nearest_clusters_df = pd.DataFrame({
    'cluster': cluster_ids,
    'nearest_cluster': [cluster_ids[i] for i in nearest_cluster],
    'centroid_cosine_similarity': nearest_similarity
}).sort_values('centroid_cosine_similarity', ascending=False)

display(nearest_clusters_df)

In [ ]:
def get_top_examples(df, cluster_id, text_col='narrative', score_col='ksil_score', n=10):
    return (
        df[df['narratives_label'] == cluster_id]
        .sort_values(score_col, ascending=False)
        [[text_col, score_col]]
        .head(n)
        .reset_index(drop=True)
    )

# Examples
cluster_a = 1
cluster_b = int(nearest_clusters_df.loc[nearest_clusters_df['cluster'] == cluster_a, 'nearest_cluster'].iloc[0])

print(f"Cluster {cluster_a} vs nearest cluster {cluster_b}")

display(get_top_examples(new_df, cluster_a, n=10))
display(get_top_examples(new_df, cluster_b, n=10))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def contrastive_terms(df, cluster_a, cluster_b, text_col='narrative',
                      ngram_range=(1,2), min_df=2, top_n=20):

    subset = df[df['narratives_label'].isin([cluster_a, cluster_b])].copy()
    texts = subset[text_col].fillna("").astype(str).tolist()

    vectorizer = TfidfVectorizer(
        stop_words='english',
        ngram_range=ngram_range,
        min_df=min_df,
        max_df=0.9
    )

    X = vectorizer.fit_transform(texts)
    features = np.array(vectorizer.get_feature_names_out())

    subset = subset.reset_index(drop=True)
    mask_a = subset['narratives_label'] == cluster_a
    mask_b = subset['narratives_label'] == cluster_b

from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import pandas as pd

def contrastive_terms(df, cluster_a, cluster_b, text_col='narrative',
                      label_col='narratives_label',
                      ngram_range=(1, 2), min_df=2, top_n=20):

    subset = df[df[label_col].isin([cluster_a, cluster_b])].copy()
    subset = subset.reset_index(drop=True)

    texts = subset[text_col].fillna("").astype(str).tolist()

    vectorizer = TfidfVectorizer(
        stop_words='english',
        ngram_range=ngram_range,
        min_df=min_df,
        max_df=0.9
    )

    X = vectorizer.fit_transform(texts)
    features = np.array(vectorizer.get_feature_names_out())

    mask_a = (subset[label_col] == cluster_a).to_numpy()
    mask_b = (subset[label_col] == cluster_b).to_numpy()

    mean_a = np.asarray(X[mask_a].mean(axis=0)).ravel()
    mean_b = np.asarray(X[mask_b].mean(axis=0)).ravel()

    diff_a = mean_a - mean_b
    diff_b = mean_b - mean_a

    top_a_idx = np.argsort(diff_a)[::-1][:top_n]
    top_b_idx = np.argsort(diff_b)[::-1][:top_n]

    top_a = pd.DataFrame({
        'term': features[top_a_idx],
        'contrast_score': diff_a[top_a_idx]
    })

    top_b = pd.DataFrame({
        'term': features[top_b_idx],
        'contrast_score': diff_b[top_b_idx]
    })

    return top_a, top_b

In [ ]:
top_a, top_b = contrastive_terms(new_df, cluster_a=1, cluster_b=4, top_n=15)

print("Terms more characteristic of cluster 1:")
display(top_a)

print("Terms more characteristic of cluster 4:")
display(top_b)

In [ ]:
all_cluster_diagnostics = []

for _, row in nearest_clusters_df.iterrows():
    c1 = int(row['cluster'])
    c2 = int(row['nearest_cluster'])

    top_c1, top_c2 = contrastive_terms(new_df, c1, c2, top_n=10)

    all_cluster_diagnostics.append({
        'cluster': c1,
        'nearest_cluster': c2,
        'centroid_cosine_similarity': row['centroid_cosine_similarity'],
        'distinctive_terms_cluster': ", ".join(top_c1['term'].head(8).tolist()),
        'distinctive_terms_nearest': ", ".join(top_c2['term'].head(8).tolist())
    })

diagnostics_df = pd.DataFrame(all_cluster_diagnostics).sort_values(
    'centroid_cosine_similarity', ascending=False
)

display(diagnostics_df)

In [ ]:
def get_representative_subset(df, cluster_id, top_n=30, score_col='ksil_score'):
    return (
        df[df['narratives_label'] == cluster_id]
        .sort_values(score_col, ascending=False)
        .head(top_n)
    )

def contrastive_terms_representative(df, cluster_a, cluster_b, text_col='narrative',
                                     score_col='ksil_score', rep_n=30,
                                     ngram_range=(1,2), min_df=2, top_n=20):

    sub_a = get_representative_subset(df, cluster_a, top_n=rep_n, score_col=score_col)
    sub_b = get_representative_subset(df, cluster_b, top_n=rep_n, score_col=score_col)
    subset = pd.concat([sub_a, sub_b], axis=0).copy()

    texts = subset[text_col].fillna("").astype(str).tolist()

    vectorizer = TfidfVectorizer(
        stop_words='english',
        ngram_range=ngram_range,
        min_df=min_df,
        max_df=0.9
    )

    X = vectorizer.fit_transform(texts)
    features = np.array(vectorizer.get_feature_names_out())

    subset = subset.reset_index(drop=True)
    mask_a = (subset['narratives_label'] == cluster_a).to_numpy()
    mask_b = (subset['narratives_label'] == cluster_b).to_numpy()

    mean_a = np.asarray(X[mask_a].mean(axis=0)).ravel()
    mean_b = np.asarray(X[mask_b].mean(axis=0)).ravel()

    diff_a = mean_a - mean_b
    diff_b = mean_b - mean_a

    top_a_idx = np.argsort(diff_a)[::-1][:top_n]
    top_b_idx = np.argsort(diff_b)[::-1][:top_n]

    top_a = pd.DataFrame({
        'term': features[top_a_idx],
        'contrast_score': diff_a[top_a_idx]
    })
    top_b = pd.DataFrame({
        'term': features[top_b_idx],
        'contrast_score': diff_b[top_b_idx]
    })

    return top_a, top_b

In [ ]:
def flag_candidate_merge(similarity, top_terms_a, top_terms_b, sim_threshold=0.90):
    if similarity >= sim_threshold:
        return "review_merge"
    return "keep"

In [ ]:
def compare_clusters_report(df, cluster_a, cluster_b, text_col='narrative', score_col='ksil_score'):
    top_a, top_b = contrastive_terms_representative(df, cluster_a, cluster_b,
                                                    text_col=text_col,
                                                    score_col=score_col,
                                                    rep_n=30,
                                                    top_n=10)

    examples_a = get_top_examples(df, cluster_a, text_col=text_col, score_col=score_col, n=5)
    examples_b = get_top_examples(df, cluster_b, text_col=text_col, score_col=score_col, n=5)

    return {
        'cluster_a': cluster_a,
        'cluster_b': cluster_b,
        'distinctive_terms_a': top_a,
        'distinctive_terms_b': top_b,
        'examples_a': examples_a,
        'examples_b': examples_b
    }

In [ ]:
report = compare_clusters_report(new_df, 1, 4)

print("Distinctive terms for cluster 1")
display(report['distinctive_terms_a'])

print("Distinctive terms for cluster 4")
display(report['distinctive_terms_b'])

print("Top examples cluster 1")
display(report['examples_a'])

print("Top examples cluster 4")
display(report['examples_b'])

In [ ]:
actor_dict = {
    'government': ['government', 'state', 'administration'],
    'judiciary': ['judicial', 'judiciary', 'court', 'judge'],
    'opposition': ['opposition', 'pasok', 'syriza'],
    'ministers': ['minister', 'ministerial'],
    'eu': ['eu', 'european', 'directive', 'regulation']
}

mechanism_dict = {
    'coverup': ['cover-up', 'coverup', 'conceal', 'hide', 'tamper'],
    'obstruction': ['obstruct', 'block', 'delay', 'manipulate'],
    'negligence': ['negligence', 'neglect', 'failure', 'inaction'],
    'privatization': ['privatization', 'fragmentation', 'private'],
    'immunity': ['immunity', 'constitutional', 'shield', 'liability']
}

In [ ]:
def show_cluster_pair(df, cluster_a, cluster_b, top_n_terms=10, top_n_examples=5):
    top_a, top_b = contrastive_terms_representative(
        df, cluster_a, cluster_b, rep_n=30, top_n=top_n_terms
    )

    ex_a = get_top_examples(df, cluster_a, n=top_n_examples)
    ex_b = get_top_examples(df, cluster_b, n=top_n_examples)

    print(f"\n=== Cluster {cluster_a} distinctive terms ===")
    display(top_a)

    print(f"\n=== Cluster {cluster_b} distinctive terms ===")
    display(top_b)

    print(f"\n=== Cluster {cluster_a} top examples ===")
    display(ex_a)

    print(f"\n=== Cluster {cluster_b} top examples ===")
    display(ex_b)

In [ ]:
show_cluster_pair(new_df, 1, 4)
show_cluster_pair(new_df, 0, 9)
show_cluster_pair(new_df, 2, 5)
show_cluster_pair(new_df, 6, 3)
show_cluster_pair(new_df, 8, 4)

In [ ]:
cluster_label_map = {
    0: "Judicial and procedural obstacles to accountability",
    1: "Government cover-up of Tempi accountability",
    2: "Systemic governance failures caused preventable disaster",
    3: "Tempi crash resulted from systemic criminal failure",
    4: "Government obstructs crash investigation to evade accountability",
    5: "Negligence caused the preventable Tempi train crash",
    6: "Railway safety remains neglected and unsecured",
    7: "Legal framework shields ministers from accountability",
    8: "Opposition exploits the tragedy for political gain",
    9: "Railway privatization caused systemic safety failures"
}

In [ ]:
new_df["narrative_label_clean"] = new_df["narratives_label"].map(cluster_label_map)

### Taxonomies

In [ ]:
collapsed_top_df["narrative_label_clean"] = collapsed_top_df["narratives_label"].map(cluster_label_map)
display(collapsed_top_df[["narratives_label", "narrative_label_clean", "narrative", "mean_top_silhouette"]])

In [ ]:
pairwise_notes = pd.DataFrame([
    {
        "cluster_a": 1,
        "cluster_b": 4,
        "difference": "Cluster 1 emphasizes cover-up/concealment; Cluster 4 emphasizes obstruction of the investigation to evade accountability.",
        "decision": "keep separate"
    },
    {
        "cluster_a": 2,
        "cluster_b": 5,
        "difference": "Cluster 2 is abstract/systemic; Cluster 5 is Tempi-specific and crash-causal.",
        "decision": "keep separate"
    },
    {
        "cluster_a": 6,
        "cluster_b": 3,
        "difference": "Cluster 6 concerns ongoing railway safety neglect; Cluster 3 interprets Tempi as systemic criminal failure.",
        "decision": "keep separate"
    }
])

display(pairwise_notes)

## Summaries

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

text_col = "narrative"
label_col = "narratives_label"
score_col = "ksil_score"

cluster_profiles = (
    new_df.groupby(label_col)
    .agg(
        Size=(text_col, "count"),
        mean_silhouette=(score_col, "mean")
    )
    .reset_index()
)

cluster_profiles["share"] = 100 * cluster_profiles["Size"] / len(new_df)


top_rep = (
    new_df.sort_values(score_col, ascending=False)
    .groupby(label_col)
    .head(1)
    [[label_col, text_col, score_col]]
    .rename(columns={
        text_col: "representative_narrative",
        score_col: "representative_silhouette"
    })
)

centroids = (
    new_df
    .groupby(label_col)
    .apply(lambda g: np.mean(X_pca[g.index], axis=0))
)

centroid_matrix = np.vstack(centroids.values)
cluster_ids = centroids.index.to_list()

sim_matrix = cosine_similarity(centroid_matrix)
sim_for_nn = sim_matrix.copy()
np.fill_diagonal(sim_for_nn, -np.inf)

nearest_idx = sim_for_nn.argmax(axis=1)
nearest_similarity = sim_for_nn.max(axis=1)

nearest_clusters_df = pd.DataFrame({
    label_col: cluster_ids,
    "nearest_cluster": [cluster_ids[i] for i in nearest_idx],
    "centroid_cosine_similarity": nearest_similarity
})

def contrastive_terms(
    df,
    cluster_a,
    cluster_b,
    text_col="narrative",
    label_col="narratives_label",
    ngram_range=(1, 2),
    min_df=2,
    top_n=6
):
    subset = df[df[label_col].isin([cluster_a, cluster_b])].copy()
    subset = subset.reset_index(drop=True)

    texts = subset[text_col].fillna("").astype(str).tolist()

    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=ngram_range,
        min_df=min_df,
        max_df=0.9
    )

    X_tfidf = vectorizer.fit_transform(texts)
    features = np.array(vectorizer.get_feature_names_out())

    mask_a = (subset[label_col] == cluster_a).to_numpy()
    mask_b = (subset[label_col] == cluster_b).to_numpy()

    mean_a = np.asarray(X_tfidf[mask_a].mean(axis=0)).ravel()
    mean_b = np.asarray(X_tfidf[mask_b].mean(axis=0)).ravel()

    diff = mean_a - mean_b
    top_idx = np.argsort(diff)[::-1][:top_n]

    return ", ".join(features[top_idx])

contrastive_rows = []

for _, row in nearest_clusters_df.iterrows():
    c = int(row[label_col])
    nn = int(row["nearest_cluster"])

    terms = contrastive_terms(
        new_df,
        cluster_a=c,
        cluster_b=nn,
        text_col=text_col,
        label_col=label_col,
        top_n=6
    )

    contrastive_rows.append({
        label_col: c,
        "contrastive_terms_vs_nearest": terms
    })

contrastive_df = pd.DataFrame(contrastive_rows)

cluster_label_map = {
    0: "Judicial and procedural obstacles to accountability",
    1: "Government cover-up of Tempi accountability",
    2: "Systemic governance failures caused preventable disaster",
    3: "Tempi crash resulted from systemic criminal failure",
    4: "Government obstructs crash investigation to evade accountability",
    5: "Negligence caused the preventable Tempi train crash",
    6: "Railway safety remains neglected and unsecured",
    7: "Legal framework shields ministers from accountability",
    8: "Opposition exploits the tragedy for political gain",
    9: "Railway privatization caused systemic safety failures"
}


taxonomy_diagnostics = (
    cluster_profiles
    .merge(top_rep, on=label_col)
    .merge(nearest_clusters_df, on=label_col)
    .merge(contrastive_df, on=label_col)
)

taxonomy_diagnostics["narrative_family"] = taxonomy_diagnostics[label_col].map(cluster_label_map)
taxonomy_diagnostics["nearest_cluster_label"] = taxonomy_diagnostics["nearest_cluster"].map(cluster_label_map)

taxonomy_diagnostics = taxonomy_diagnostics[
    [
        label_col,
        "narrative_family",
        "share",
        "mean_silhouette",
        "representative_narrative",
        "nearest_cluster",
        "centroid_cosine_similarity",
        "contrastive_terms_vs_nearest"
    ]
].sort_values(label_col)

display(taxonomy_diagnostics)

In [ ]:
latex_table = taxonomy_diagnostics.copy()

latex_table["share"] = latex_table["share"].map(lambda x: f"{x:.2f}")
latex_table["mean_silhouette"] = latex_table["mean_silhouette"].map(lambda x: f"{x:.3f}")
latex_table["centroid_cosine_similarity"] = latex_table["centroid_cosine_similarity"].map(lambda x: f"{x:.3f}")

print(
    latex_table.to_latex(
        index=False,
        escape=True,
        column_format="p{0.04\\linewidth} p{0.22\\linewidth} p{0.07\\linewidth} p{0.08\\linewidth} p{0.27\\linewidth} p{0.07\\linewidth} p{0.08\\linewidth} p{0.17\\linewidth}",
        caption="Representative and contrastive characterization of the final narrative clusters.",
        label="tab:taxonomy_diagnostics"
    )
)

In [ ]:
top_n = 3

top_reps = (
    new_df.sort_values(score_col, ascending=False)
    .groupby(label_col)
    .head(top_n)
    .copy()
)

top_reps["rank"] = (
    top_reps.groupby(label_col)[score_col]
    .rank(method="first", ascending=False)
    .astype(int)
)

top_reps["narrative_family"] = top_reps[label_col].map(cluster_label_map)

top_reps_table = top_reps[
    [label_col, "narrative_family", "rank", text_col, score_col]
].sort_values([label_col, "rank"])

display(top_reps_table)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

centroids = (
    new_df
    .groupby(label_col)
    .apply(lambda g: np.mean(X_pca[g.index], axis=0))
)

centroid_matrix = np.vstack(centroids.values)
cluster_ids = centroids.index.to_list()

sim_matrix = cosine_similarity(centroid_matrix)

sim_df = pd.DataFrame(
    sim_matrix,
    index=[f"C{c}" for c in cluster_ids],
    columns=[f"C{c}" for c in cluster_ids]
)

plt.figure(figsize=(8, 6))
sns.heatmap(
    sim_df,
    annot=True,
    fmt=".2f",
    cmap="Reds",
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Centroid cosine similarity"}
)

plt.title("")
plt.tight_layout()
plt.show()

In [ ]:
sim_matrix = cosine_similarity(centroid_matrix)

mask = np.triu(np.ones_like(sim_matrix, dtype=bool), k=1)

sim_df = pd.DataFrame(
    sim_matrix,
    index=[f"C{c}" for c in cluster_ids],
    columns=[f"C{c}" for c in cluster_ids]
)

plt.figure(figsize=(8, 6))
sns.heatmap(
    sim_df,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="Reds",
    square=True,
    linewidths=0.5,
    cbar_kws={"label": ""}
)

plt.title("")
plt.tight_layout()
plt.savefig("centroidsimilarity.png", dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


sim_matrix = cosine_similarity(centroid_matrix)

mask = np.triu(np.ones_like(sim_matrix, dtype=bool), k=1)

sim_df = pd.DataFrame(
    sim_matrix,
    index=[f"C{c}" for c in cluster_ids],
    columns=[f"C{c}" for c in cluster_ids]
)

plt.figure(figsize=(10, 8))

sns.heatmap(
    sim_df,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="Reds",
    square=True,
    linewidths=0.5,

    annot_kws={"size": 12, "weight": "bold"},
    cbar_kws={"label": ""}
)

plt.xticks(fontsize=12, fontweight='bold')
plt.yticks(fontsize=12, fontweight='bold')

plt.title("")
plt.tight_layout()
plt.savefig("centroidsimilarity.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)


label_col = "narratives_label"
text_col = "narrative"
score_col = "ksil_score"

TOP_REPRESENTATIVES = 1

TOP_TERMS = 8

cluster_profiles = (
    new_df
    .groupby(label_col)
    .agg(
        Size=(text_col, "count"),
        mean_silhouette=(score_col, "mean")
    )
    .reset_index()
)

cluster_profiles["share_percent"] = (
    cluster_profiles["Size"] / len(new_df) * 100
)

cluster_profiles["narrative_family"] = (
    cluster_profiles[label_col].map(cluster_label_map)
)


top_reps_long = (
    new_df
    .sort_values([label_col, score_col], ascending=[True, False])
    .groupby(label_col)
    .head(TOP_REPRESENTATIVES)
    .copy()
)

top_reps_long["rep_rank"] = (
    top_reps_long
    .groupby(label_col)
    .cumcount() + 1
)

def collapse_representatives(group):
    rows = []
    for _, row in group.iterrows():
        rows.append(
            f"{int(row['rep_rank'])}. {row[text_col]} "
            f"(sil={row[score_col]:.3f})"
        )
    return "\n".join(rows)

top_reps = (
    top_reps_long
    .groupby(label_col)
    .apply(collapse_representatives)
    .reset_index(name="representative_narratives")
)

centroids = (
    new_df
    .groupby(label_col)
    .apply(lambda g: np.mean(X_pca[g.index], axis=0))
)

cluster_ids = centroids.index.to_list()
centroid_matrix = np.vstack(centroids.values)

sim_matrix = cosine_similarity(centroid_matrix)

sim_df = pd.DataFrame(
    sim_matrix,
    index=cluster_ids,
    columns=cluster_ids
)

sim_for_nn = sim_matrix.copy()
np.fill_diagonal(sim_for_nn, -np.inf)

nearest_idx = sim_for_nn.argmax(axis=1)
nearest_similarity = sim_for_nn.max(axis=1)

nearest_clusters_df = pd.DataFrame({
    label_col: cluster_ids,
    "nearest_cluster": [cluster_ids[i] for i in nearest_idx],
    "centroid_cosine_similarity": nearest_similarity
})

nearest_clusters_df["nearest_cluster_label"] = (
    nearest_clusters_df["nearest_cluster"].map(cluster_label_map)
)


def get_contrastive_terms(
    df,
    cluster_a,
    cluster_b,
    text_col="narrative",
    label_col="narratives_label",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.90,
    top_n=8,
    use_representative_subset=False,
    rep_n=50,
    score_col="ksil_score"
):
    """
    Returns terms that are more characteristic of cluster_a than cluster_b.

    If use_representative_subset=True, TF-IDF is computed only on the top
    silhouette-ranked narratives from each cluster. This can make terms
    cleaner and more interpretable.
    """

    if use_representative_subset:
        sub_a = (
            df[df[label_col] == cluster_a]
            .sort_values(score_col, ascending=False)
            .head(rep_n)
        )
        sub_b = (
            df[df[label_col] == cluster_b]
            .sort_values(score_col, ascending=False)
            .head(rep_n)
        )
        subset = pd.concat([sub_a, sub_b], axis=0).copy()
    else:
        subset = df[df[label_col].isin([cluster_a, cluster_b])].copy()

    subset = subset.reset_index(drop=True)
    texts = subset[text_col].fillna("").astype(str).tolist()

    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=ngram_range,
        min_df=min_df,
        max_df=max_df
    )

    X_tfidf = vectorizer.fit_transform(texts)
    features = np.array(vectorizer.get_feature_names_out())

    mask_a = (subset[label_col] == cluster_a).to_numpy()
    mask_b = (subset[label_col] == cluster_b).to_numpy()

    mean_a = np.asarray(X_tfidf[mask_a].mean(axis=0)).ravel()
    mean_b = np.asarray(X_tfidf[mask_b].mean(axis=0)).ravel()

    diff = mean_a - mean_b
    top_idx = np.argsort(diff)[::-1][:top_n]

    terms = features[top_idx]
    scores = diff[top_idx]

    terms_df = pd.DataFrame({
        "term": terms,
        "contrast_score": scores
    })

    return terms_df


contrastive_rows = []

for _, row in nearest_clusters_df.iterrows():
    c = int(row[label_col])
    nn = int(row["nearest_cluster"])

    terms_df = get_contrastive_terms(
        new_df,
        cluster_a=c,
        cluster_b=nn,
        text_col=text_col,
        label_col=label_col,
        top_n=TOP_TERMS,
        use_representative_subset=False
    )

    contrastive_rows.append({
        label_col: c,
        "contrastive_terms_vs_nearest": ", ".join(terms_df["term"].tolist())
    })

contrastive_df = pd.DataFrame(contrastive_rows)

taxonomy_diagnostics = (
    cluster_profiles
    .merge(top_reps, on=label_col, how="left")
    .merge(nearest_clusters_df, on=label_col, how="left")
    .merge(contrastive_df, on=label_col, how="left")
    .sort_values(label_col)
    .reset_index(drop=True)
)

taxonomy_diagnostics = taxonomy_diagnostics[
    [
        label_col,
        "narrative_family",
        "Size",
        "share_percent",
        "mean_silhouette",
        "representative_narratives",
        "nearest_cluster",
        "nearest_cluster_label",
        "centroid_cosine_similarity",
        "contrastive_terms_vs_nearest"
    ]
]

display(taxonomy_diagnostics)

taxonomy_table_paper = taxonomy_diagnostics.copy()

taxonomy_table_paper["share_percent"] = taxonomy_table_paper["share_percent"].map(
    lambda x: f"{x:.2f}"
)
taxonomy_table_paper["mean_silhouette"] = taxonomy_table_paper["mean_silhouette"].map(
    lambda x: f"{x:.3f}"
)
taxonomy_table_paper["centroid_cosine_similarity"] = taxonomy_table_paper["centroid_cosine_similarity"].map(
    lambda x: f"{x:.3f}"
)

taxonomy_table_paper = taxonomy_table_paper.rename(columns={
    label_col: "Cluster",
    "narrative_family": "Narrative family",
    "Size": "Size",
    "share_percent": "Share (%)",
    "mean_silhouette": "Mean silhouette",
    "representative_narratives": "Representative narrative(s)",
    "nearest_cluster": "Nearest cluster",
    "nearest_cluster_label": "Nearest cluster label",
    "centroid_cosine_similarity": "Cosine similarity",
    "contrastive_terms_vs_nearest": "Contrastive TF-IDF terms"
})

display(taxonomy_table_paper)

taxonomy_table_paper.to_csv(
    "taxonomy_diagnostics_table.csv",
    index=False
)

latex_code = taxonomy_table_paper.to_latex(
    index=False,
    escape=True,
    column_format=(
        "p{0.04\\linewidth} "
        "p{0.18\\linewidth} "
        "p{0.05\\linewidth} "
        "p{0.06\\linewidth} "
        "p{0.07\\linewidth} "
        "p{0.25\\linewidth} "
        "p{0.05\\linewidth} "
        "p{0.07\\linewidth} "
        "p{0.20\\linewidth}"
    ),
    columns=[
        "Cluster",
        "Narrative family",
        "Size",
        "Share (%)",
        "Mean silhouette",
        "Representative narrative(s)",
        "Nearest cluster",
        "Cosine similarity",
        "Contrastive TF-IDF terms"
    ],
    caption=(
        "Representative and contrastive characterization of the final narrative clusters. "
        "Representative narratives are selected using centroid-approximated silhouette scores. "
        "Contrastive TF-IDF terms are computed against each cluster's nearest neighboring cluster."
    ),
    label="tab:taxonomy_diagnostics"
)

print(latex_code)

In [ ]:
# ============================================================
# PUBLICATION-STYLE CONTRASTIVE TF-IDF FIGURE
# Cluster vs all other clusters
# Small multiples: top terms per narrative cluster
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
import textwrap

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

label_col = "narratives_label"
text_col = "narrative"

TOP_TERMS = 6

# Keep cluster order fixed
cluster_ids = sorted(new_df[label_col].dropna().unique())

# ------------------------------------------------------------
# Compute contrastive TF-IDF terms: cluster vs all others
# ------------------------------------------------------------

def compute_contrastive_tfidf_vs_rest(
    df,
    cluster_id,
    text_col="narrative",
    label_col="narratives_label",
    top_terms=6,
    min_df=2,
    max_df=0.90,
    ngram_range=(1, 2)
):
    work_df = df.copy().reset_index(drop=True)

    texts = work_df[text_col].fillna("").astype(str).tolist()

    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=ngram_range,
        min_df=min_df,
        max_df=max_df
    )

    X_tfidf = vectorizer.fit_transform(texts)
    features = np.array(vectorizer.get_feature_names_out())

    mask_c = (work_df[label_col] == cluster_id).to_numpy()
    mask_rest = ~mask_c

    mean_c = np.asarray(X_tfidf[mask_c].mean(axis=0)).ravel()
    mean_rest = np.asarray(X_tfidf[mask_rest].mean(axis=0)).ravel()

    contrast = mean_c - mean_rest

    top_idx = np.argsort(contrast)[::-1][:top_terms]

    out = pd.DataFrame({
        "cluster": cluster_id,
        "term": features[top_idx],
        "contrast_score": contrast[top_idx]
    })

    return out


contrastive_terms_long = []

for c in cluster_ids:
    contrastive_terms_long.append(
        compute_contrastive_tfidf_vs_rest(
            new_df,
            cluster_id=c,
            text_col=text_col,
            label_col=label_col,
            top_terms=TOP_TERMS,
            min_df=2,
            max_df=0.90,
            ngram_range=(1, 2)
        )
    )

contrastive_terms_long = pd.concat(contrastive_terms_long, ignore_index=True)

contrastive_terms_long["narrative_family"] = (
    contrastive_terms_long["cluster"].map(cluster_label_map)
)

display(contrastive_terms_long)

In [ ]:
# ============================================================
# EXACTLY 3 UNIQUE DISTINCTIVE REPRESENTATIVES PER CLUSTER
# Cluster vs all other clusters
# Strong duplicate / near-duplicate filtering
# ============================================================

import re
import string
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

label_col = "narratives_label"
text_col = "narrative"
score_col = "ksil_score"

TOP_REPRESENTATIVES = 3
TOP_TERMS = 8

# Similarity threshold for near-duplicate representatives
# Higher = stricter only for almost identical strings
# Lower = removes more semantically similar items
NEAR_DUP_THRESHOLD = 0.88

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)


# ------------------------------------------------------------
# 1. Strong text normalization
# ------------------------------------------------------------

def normalize_narrative_text(x):
    """
    Collapses duplicates caused by capitalization, punctuation, and spacing.
    Example:
    'Government cover-up obstructs accountability.'
    and
    'government cover up obstructs accountability'
    become close/equivalent.
    """
    text = str(x).lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text


# ------------------------------------------------------------
# 2. Compute contrastive TF-IDF: cluster vs all other clusters
# ------------------------------------------------------------

def get_cluster_vs_rest_contrastive_info(
    df,
    cluster_id,
    text_col="narrative",
    label_col="narratives_label",
    score_col="ksil_score",
    top_terms=8,
    min_df=2,
    max_df=0.90,
    ngram_range=(1, 2)
):
    """
    Computes contrastive TF-IDF information for one cluster against all other clusters.

    Returns:
    - terms_df: top contrastive TF-IDF terms for this cluster
    - cluster_subset: candidate narratives from this cluster with scores
    - vectorizer_cluster: TF-IDF vectorizer fitted on normalized cluster narratives
    - X_cluster_text: TF-IDF matrix for normalized cluster narratives
    """

    work_df = df.copy().reset_index(drop=True)

    if text_col not in work_df.columns:
        raise ValueError(f"Column '{text_col}' not found in dataframe.")
    if label_col not in work_df.columns:
        raise ValueError(f"Column '{label_col}' not found in dataframe.")
    if score_col not in work_df.columns:
        raise ValueError(f"Column '{score_col}' not found in dataframe.")

    texts = work_df[text_col].fillna("").astype(str).tolist()

    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=ngram_range,
        min_df=min_df,
        max_df=max_df
    )

    X_tfidf = vectorizer.fit_transform(texts)
    features = np.array(vectorizer.get_feature_names_out())

    mask_c = (work_df[label_col] == cluster_id).to_numpy()
    mask_rest = ~mask_c

    if mask_c.sum() == 0:
        raise ValueError(f"No rows found for cluster {cluster_id}.")
    if mask_rest.sum() == 0:
        raise ValueError("Cannot compute contrastive terms with only one cluster.")

    mean_c = np.asarray(X_tfidf[mask_c].mean(axis=0)).ravel()
    mean_rest = np.asarray(X_tfidf[mask_rest].mean(axis=0)).ravel()

    # Positive values mean: more characteristic of this cluster than all others
    diff = mean_c - mean_rest

    top_idx = np.argsort(diff)[::-1][:top_terms]

    terms_df = pd.DataFrame({
        "term": features[top_idx],
        "contrast_score": diff[top_idx]
    })

    cluster_subset = work_df.loc[mask_c].copy()

    # Lexical distinctiveness of each narrative:
    # sum of its TF-IDF weights on terms that are positively contrastive for this cluster.
    positive_diff = np.clip(diff, 0, None)

    lexical_distinctiveness = np.asarray(
        X_tfidf[mask_c].multiply(positive_diff).sum(axis=1)
    ).ravel()

    cluster_subset["lexical_distinctiveness"] = lexical_distinctiveness

    # Combined score:
    # centrality x distinctiveness
    cluster_subset["distinctive_rep_score"] = (
        cluster_subset[score_col] * cluster_subset["lexical_distinctiveness"]
    )

    cluster_subset["normalized_narrative"] = cluster_subset[text_col].map(
        normalize_narrative_text
    )

    return terms_df, cluster_subset


# ------------------------------------------------------------
# 3. Select exactly 3 unique / non-near-duplicate representatives
# ------------------------------------------------------------

def select_exactly_three_distinctive_reps(
    cluster_subset,
    text_col="narrative",
    score_col="distinctive_rep_score",
    top_n=3,
    near_dup_threshold=0.88
):
    """
    Selects exactly top_n representatives if possible.

    Selection logic:
    1. Sort by distinctive_rep_score.
    2. Remove exact duplicates after normalization.
    3. Remove near-duplicates using TF-IDF cosine similarity.
    4. If fewer than top_n remain, relax the near-duplicate threshold.
    5. If still fewer than top_n, fill with next unique normalized narratives.
    """

    candidates = (
        cluster_subset
        .sort_values(score_col, ascending=False)
        .drop_duplicates(subset=["normalized_narrative"])
        .copy()
    )

    if len(candidates) == 0:
        return candidates

    # If fewer/equal than top_n unique exact candidates, return them
    if len(candidates) <= top_n:
        return candidates.head(top_n).copy()

    # Fit TF-IDF on normalized narratives inside this cluster
    normalized_texts = candidates["normalized_narrative"].fillna("").astype(str).tolist()

    # Use character n-grams to catch near-duplicates differing slightly in wording/punctuation
    dup_vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=1
    )

    X_dup = dup_vectorizer.fit_transform(normalized_texts)

    selected_indices = []

    def greedy_select(threshold):
        selected = []

        for i in range(len(candidates)):
            if len(selected) == 0:
                selected.append(i)
            else:
                sims = cosine_similarity(X_dup[i], X_dup[selected]).ravel()
                if sims.max() < threshold:
                    selected.append(i)

            if len(selected) == top_n:
                break

        return selected

    # Try initial threshold, then relax if needed
    thresholds = [
        near_dup_threshold,
        0.92,
        0.95,
        0.98,
        1.01  # effectively only exact duplicates removed
    ]

    for threshold in thresholds:
        selected_indices = greedy_select(threshold)
        if len(selected_indices) >= top_n:
            break

    selected = candidates.iloc[selected_indices].copy()

    # Final fallback: if still fewer than top_n, fill with next exact-unique candidates
    if len(selected) < top_n:
        already_norm = set(selected["normalized_narrative"].tolist())

        fillers = candidates[
            ~candidates["normalized_narrative"].isin(already_norm)
        ].head(top_n - len(selected))

        selected = pd.concat([selected, fillers], axis=0)

    return selected.head(top_n).copy()


# ------------------------------------------------------------
# 4. Build final representatives and contrastive terms
# ------------------------------------------------------------

contrastive_rows = []
distinctive_rep_rows = []

cluster_ids = sorted(new_df[label_col].dropna().unique())

for c in cluster_ids:

    terms_df, cluster_subset = get_cluster_vs_rest_contrastive_info(
        new_df,
        cluster_id=c,
        text_col=text_col,
        label_col=label_col,
        score_col=score_col,
        top_terms=TOP_TERMS,
        min_df=2,
        max_df=0.90,
        ngram_range=(1, 2)
    )

    contrastive_rows.append({
        label_col: c,
        "contrastive_terms_vs_all_other_clusters": ", ".join(terms_df["term"].tolist())
    })

    selected = select_exactly_three_distinctive_reps(
        cluster_subset,
        text_col=text_col,
        score_col="distinctive_rep_score",
        top_n=TOP_REPRESENTATIVES,
        near_dup_threshold=NEAR_DUP_THRESHOLD
    )

    selected["rep_rank"] = range(1, len(selected) + 1)

    # Check if fewer than 3 were found
    if len(selected) < TOP_REPRESENTATIVES:
        print(
            f"Warning: Cluster {c} has only {len(selected)} unique representatives "
            f"after duplicate filtering."
        )

    for _, row in selected.iterrows():
        distinctive_rep_rows.append({
            label_col: c,
            "narrative_family": cluster_label_map.get(c, str(c)),
            "rep_rank": int(row["rep_rank"]),
            "representative_narrative": row[text_col],
            "silhouette": row[score_col],
            "lexical_distinctiveness": row["lexical_distinctiveness"],
            "distinctive_rep_score": row["distinctive_rep_score"]
        })


contrastive_df = pd.DataFrame(contrastive_rows)
distinctive_reps_long = pd.DataFrame(distinctive_rep_rows)


# ------------------------------------------------------------
# 5. Display long representatives table for inspection
# ------------------------------------------------------------

distinctive_reps_long_display = distinctive_reps_long[
    [
        label_col,
        "narrative_family",
        "rep_rank",
        "representative_narrative",
        "silhouette",
        "lexical_distinctiveness",
        "distinctive_rep_score"
    ]
].sort_values([label_col, "rep_rank"])

display(distinctive_reps_long_display)


# ------------------------------------------------------------
# 6. Collapse representatives into one cell per cluster
# ------------------------------------------------------------

def collapse_distinctive_reps(group):
    group = group.sort_values("rep_rank")
    lines = []

    for _, row in group.iterrows():
        lines.append(
            f"{int(row['rep_rank'])}. {row['representative_narrative']}"
        )

    return "\n\n".join(lines)


distinctive_reps_collapsed = (
    distinctive_reps_long
    .groupby(label_col)
    .apply(collapse_distinctive_reps)
    .reset_index(name="top_3_unique_distinctive_representatives")
)


# ------------------------------------------------------------
# 7. Final table: terms + representatives
# ------------------------------------------------------------

final_taxonomy_contrastive_unique = (
    pd.DataFrame({label_col: cluster_ids})
    .merge(contrastive_df, on=label_col, how="left")
    .merge(distinctive_reps_collapsed, on=label_col, how="left")
)

final_taxonomy_contrastive_unique["narrative_family"] = (
    final_taxonomy_contrastive_unique[label_col].map(cluster_label_map)
)

final_taxonomy_contrastive_unique = final_taxonomy_contrastive_unique[
    [
        label_col,
        "narrative_family",
        "contrastive_terms_vs_all_other_clusters",
        "top_3_unique_distinctive_representatives"
    ]
].rename(columns={
    label_col: "Cluster",
    "narrative_family": "Narrative family",
    "contrastive_terms_vs_all_other_clusters": "Contrastive TF-IDF terms",
    "top_3_unique_distinctive_representatives": "Top 3 unique distinctive representatives"
})

display(final_taxonomy_contrastive_unique)


# ------------------------------------------------------------
# 8. Export
# ------------------------------------------------------------

distinctive_reps_long_display.to_csv(
    "distinctive_representatives_long_with_scores.csv",
    index=False
)

final_taxonomy_contrastive_unique.to_csv(
    "final_taxonomy_contrastive_unique_exact3.csv",
    index=False
)


latex_code = final_taxonomy_contrastive_unique[
    [
        "Narrative family",
        "Contrastive TF-IDF terms",
        "Top 3 unique distinctive representatives"
    ]
].to_latex(
    index=False,
    escape=True,
    column_format=(
        "p{0.28\\linewidth} "
        "p{0.27\\linewidth} "
        "p{0.39\\linewidth}"
    ),
    caption=(
        "Contrastive characterization of the final narrative clusters. "
        "Contrastive TF-IDF terms and distinctive representative narratives are computed "
        "by comparing each cluster against all other clusters. Representatives are selected "
        "using a combined criterion that rewards both silhouette-based centrality and lexical "
        "distinctiveness, while duplicate and near-duplicate statements are collapsed."
    ),
    label="tab:taxonomy_contrastive_unique"
)

print(latex_code)